In [1]:
from pathlib import Path
import hashlib
import pandas as pd
base = Path.home() / "IAA" / "data_part2"
for f in sorted((base / "development").glob("*.parquet")):
    g = base / "public1" / f.name
    same = g.exists() and hashlib.md5(f.read_bytes()).hexdigest() == hashlib.md5(g.read_bytes()).hexdigest()
    print(f"{f.name:28s} identical: {same}")
for folder in ["development", "public1"]:
    c = pd.read_parquet(base / folder / "claims.parquet", columns=["loss_date", "report_date"])
    print(folder, "claims:", len(c), "| loss dates", c.loss_date.min(), "to", c.loss_date.max(),
          "| last report", c.report_date.max())

addresses.parquet            identical: True
claims.parquet               identical: True
customers.parquet            identical: True
policies_buildings.parquet   identical: True
policies_contents.parquet    identical: True
policies_motor.parquet       identical: True
sample_submission.parquet    identical: True
test_buildings.parquet       identical: True
test_contents.parquet        identical: True
test_motor.parquet           identical: True
weather.parquet              identical: True
development claims: 163150 | loss dates 2020-12-01 to 2024-12-29 | last report 2024-12-30
public1 claims: 163150 | loss dates 2020-12-01 to 2024-12-29 | last report 2024-12-30


In [2]:
from pathlib import Path
a = Path.home() / "IAA" / "AI_USE_LOG.md"
ROWS = {
"| 102 |": """| 102 | E | When buildings fire was moved to a flat rate, AI kept applying the Part 1 tail condition to all perils. In the safest decile the flat rate is most of the prediction, so the condition measured fire luck rather than ranking. | Noticed after E12 failed the condition; the team asked why fire had been included. | Rule changed by the team lead (tails on the modelled part; D² and A/E stay on all perils), disclosed in DECISIONS and applied to every later comparison. E12 re-judged: C adopted. |
""",
"| 103 |": """| 103 | E | AI set up Part 2 to learn every relativity from the emerging data alone (one tenth of the developed data; one training year in the fold) and used the developed market only for trends, the tail and a new-business comparison. It did not question this design until the leaderboard showed a gap of 0.05. A day of tests went to other causes (flood, weather, marketing variables, premium; each 0–0.004). | The team's leaderboard check; then the black box on emerging data only (0.0757) pointed away from features and model structure; E17 tested developed data as training data. | Pooled validation D² 0.0806 → 0.1096; public 0.11185 → 0.12187. Lesson: when a market is small, ask first which data can carry the relativities, before tuning on the small sample. |
""",
}
s = a.read_text()
lines = s.splitlines(keepends=True)
k = max(n for n, l in enumerate(lines) if l.startswith("| 101 |"))
add = "".join(v for key, v in ROWS.items() if key not in s)
a.write_text("".join(lines[:k + 1]) + add + "".join(lines[k + 1:]))
print("added:", [key for key in ROWS if key not in s] or "nothing (already there)")

added: ['| 102 |', '| 103 |']


In [3]:
from pathlib import Path
d = Path.home() / "IAA" / "DECISIONS.md"
E18 = """### E18: weight of developed rows in combined training (Oct 24) — no change [notebook 23]
- Rule (fixed before running): candidates per line chosen by 3-fold CV inside emerging 2023 (developed rows always in
  training, level set on the emerging rows); adopted if 2024 D² is at least +0.0005 over v3 and the tail condition holds.
- Motor: CV picks developed only (current, CV 0.0430; combined weights 1 / 0.3 / 0.1: 0.0426 / 0.0413 / 0.0381) — kept.
- Contents: CV picks developed weight 0.1 (CV 0.0783 vs 0.0754): 2024 D² 0.0647 vs 0.0638 (+0.0009), but the highest
  decile moves from 1.03 to 0.83 (0.14 further from 1; limit 0.1) — kept at weight 1.
- Buildings other perils: CV picks combined weight 1 (2024 D² 0.0991 vs 0.0720) = E17b's B3, which fails the lowest
  decile under rule b (0.46 vs 0.89) — kept emerging only. (The notebook's buildings tails were computed on the other
  perils only; the decision uses the rule-b figures from E17b.) Third time the same result: combining the other
  buildings perils raises D² by about 0.02–0.03 but over-prices the safest emerging homes.

"""
s = d.read_text()
assert "### E18:" not in s
i = s.find("## Assumptions register")
d.write_text(s[:i] + E18 + s[i:])
print("E18 inserted before ## Assumptions register")

E18 inserted before ## Assumptions register


AssertionError: 

In [5]:
from pathlib import Path
s = (Path.home()/"IAA"/"DECISIONS.md").read_text()
print("E21:", s.count("### E21:"), "| Blind design:", s.count("### Blind design by a second AI"))

E21: 1 | Blind design: 1
